# R24 free-gate adjudication - closing the loss classes for free

**Author**: Claude (opus executor)  
**Date**: 2026-07-08  
**Purpose**: Adjudicate the three free (zero-LLM, CPU-only) clauses of the R24 remedies tier - H252 (scanner-audited residue), H253 (severed-association linkage), H254 (graph-as-lexicon healing ceiling) - entirely from existing disk artifacts plus one read-only reference-graph query set. No LLM calls, no GPU, no database writes. Each clause is scored against the bar pre-registered in `docs/experiments/kgf-redesign-experiments.md` section R24.

The corpus is a technical benchmark document set (CPAP device datasheets and manuals); all analysis is lexical / statistical over the frozen H119 extraction checkpoints, the H144 boundary-audit method, the H226 ownership census, and the read-only reference graph.

**Frozen inputs**
- `results/h119/` - 150 checkpoints (3 arms x 5 runs x 10 docs); arm `A_production` is the production recipe
- `data/interim/h119_chunks.pkl` - exact chunk texts per document
- `data/processed/probes-wide-v2-h195.json` - probed products (gold carriers), field `product`
- `reports/ownership-census-h226-20260707T204912Z.json` - H226 never-extracted ownership misses
- `data/external/cpap-datasheets-and-manuals/*.pdf` - source PDFs for the H144 severed-row recompute
- reference graph `bolt://172.19.0.9:7687` (READ-ONLY) - entity names + `source_documents`

**Frozen gold-carrier harness (reused exactly from the R23 notebook)**: a name matches a probed product when `rapidfuzz.fuzz.token_set_ratio(name.lower(), product.lower()) >= 85`. Union-of-5 (arm A) = all products matched by any of the 5 arm-A runs pooled over the 10 documents. Sanity anchors: union5 = 63, single-run mean coverage = 76.8%, H248 scanner 55/63.

## Imports

In [1]:
# Imports - grouped by category
import os                                         # cwd normalization under nbconvert
import re                                         # H248 scanner + sentence split
import json                                       # artifacts + report serialization
import glob                                       # checkpoint discovery
import pickle                                     # chunk cache
import itertools                                  # (unused-safe) pair helpers
from statistics import mean                       # metric aggregation
from collections import Counter                   # per-doc tallies
from datetime import datetime, timezone           # UTC report stamp
from pathlib import Path                          # filesystem paths

import tiktoken                                   # cl100k chunk-boundary recompute (H144 method)
from rapidfuzz import fuzz                        # frozen token_set_ratio matcher
from neo4j import GraphDatabase                   # READ-ONLY reference graph (H254)
from loguru import logger                         # silence reader DEBUG spam
from rich.console import Console                  # config + result rendering
from rich.table import Table
from rich import box

from knowledge_graph_foundry.models import normalize_name             # frozen name normalization
from knowledge_graph_foundry.config import PROJ_ROOT                  # canonical project root
from knowledge_graph_foundry.ingest.readers import read_document      # H144 severed-row recompute
from knowledge_graph_foundry.ingest.chunking import _snap_to_boundary # H144 chunk boundary

os.chdir(PROJ_ROOT)
logger.remove()                                   # reader logs at DEBUG - drop them
console = Console()
print("imports ok; cwd:", os.getcwd())

imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration

Frozen parameters, artifact load, and the R23 sanity anchors. The anchors MUST reproduce the frozen reference numbers before any adjudication is trusted.

In [2]:
# --- Frozen configuration ---
THR = 85                                          # token_set_ratio threshold (gold carriers, name-in-text, graph match)
CKPT_GLOB = "results/h119/*.json"
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
PROBES = Path("data/processed/probes-wide-v2-h195.json")
H226_REPORT = Path("reports/ownership-census-h226-20260707T204912Z.json")
DOC_DIR = Path("data/external/cpap-datasheets-and-manuals")
CHUNK_SIZE, CHUNK_OVERLAP = 2000, 200             # H144 chunker parameters
NEO4J_URI = "bolt://172.19.0.9:7687"              # READ-ONLY reference graph
NEO4J_AUTH = ("neo4j", "kgfoundry")
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(exist_ok=True)
LOG_PATH = Path("logs/r24-free-gates.log")

BAR_H252_SHARE, BAR_H252_COST = 0.60, 0.40        # residue recoverable share / token-cost fraction
BAR_H253_LINK, H253_MATCH_THR = 0.25, 70          # severance-linkage share / feature-in-row token_set_ratio
BAR_H254_CEILING = 0.50                           # graph-lexicon healing ceiling

def log(msg: str) -> None:
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with open(LOG_PATH, "a") as fh:
        fh.write(f"[{stamp}] {msg}\n")

# --- Load checkpoints: arm -> run -> doc -> [names] ---
ARMS = {}
for f in glob.glob(CKPT_GLOB):
    d = json.load(open(f))
    ARMS.setdefault(d["arm"], {}).setdefault(d["run"], {})[d["doc"]] = d["names"]
A = ARMS["A_production"]
RUNS = sorted(A); DOCS = sorted(A[1])

# --- Gold carriers ---
PRODUCTS = list(dict.fromkeys(p["product"] for p in json.load(open(PROBES))["probes"]))

# --- Chunk texts: doc -> concatenated chunk text (index order) ---
chunks = pickle.load(open(CHUNK_CACHE, "rb"))
DOCTEXT = {doc: "".join(c["text"] for c in sorted(chunks[doc], key=lambda c: c["index"]))
           for doc in DOCS}

# --- Frozen matcher ---
def matched(ent_list):
    el = [e.lower() for e in ent_list]
    return {i for i, prod in enumerate(PRODUCTS)
            if any(fuzz.token_set_ratio(e, prod.lower()) >= THR for e in el)}
def run_ents(r):
    return [n for doc in DOCS for n in A[r].get(doc, [])]

perA = {r: matched(run_ents(r)) for r in RUNS}
UNION5 = set().union(*perA.values()); U = len(UNION5)
single_cov = mean(len(perA[r]) / U for r in RUNS)

RESULTS = {}                                       # per-hypothesis metrics for the report

t = Table(title="Configuration and sanity anchors", box=box.SIMPLE, show_header=True)
t.add_column("key"); t.add_column("value"); t.add_column("expected")
t.add_row("arms x runs x docs", f"{len(ARMS)} x {len(RUNS)} x {len(DOCS)}", "3 x 5 x 10")
t.add_row("gold products (unique)", str(len(PRODUCTS)), "101")
t.add_row("union-of-5 coverable", str(U), "63")
t.add_row("single-run mean coverage", f"{single_cov:.1%}", "76.8%")
console.print(t)
log(f"config: union5={U} single_cov={single_cov:.3f}")

           Configuration and sanity anchors           
                                                      
  key                        value        expected    
 ──────────────────────────────────────────────────── 
  arms x runs x docs         3 x 5 x 10   3 x 5 x 10  
  gold products (unique)     101          101         
  union-of-5 coverable       63           63          
  single-run mean coverage   76.8%        76.8%

## Shared harness - the H248 scanner and per-document localization

The H248 scanner is three deterministic patterns (alphanumeric model codes, capitalized multiword forms, all-caps brand tokens) applied to chunk text. A carrier is **scanner-visible** if some candidate surface form matches it (`token_set_ratio >= 85`). For localization, chunk text is split into sentences (on newline or period); a carrier is **span-locatable** in a document when a sentence there contains a candidate that matches it. Because every candidate lives inside a sentence, scanner-visibility and span-locatability coincide at document granularity - a carrier is recoverable iff it has at least one span-sentence somewhere.

In [3]:
# --- H248 scanner patterns (reproduced exactly from the R23 notebook) ---
CODE_RX  = re.compile(r"\b(?=[A-Za-z0-9\-]*[A-Za-z])(?=[A-Za-z0-9\-]*\d)[A-Za-z][A-Za-z0-9\-]{1,}\b")
MULTI_RX = re.compile(r"\b[A-Z][a-zA-Z]+(?:\s+(?:[A-Z][a-zA-Z0-9]*|[A-Z0-9]{2,}|\d+[A-Za-z]*)){1,5}\b")
CAPS_RX  = re.compile(r"\b[A-Z]{2,}[A-Z0-9]*\b")
RXS = (CODE_RX, MULTI_RX, CAPS_RX)

def sentences(txt):
    return re.split(r"[\n.]", txt)

# global scanner-visible set (H248 clause-1 reproduction: candidates pooled over all docs)
_gcands = set()
for doc in DOCS:
    for rx in RXS:
        for m in rx.finditer(DOCTEXT[doc]):
            _gcands.add(m.group(0).strip().lower())
covered = sorted(UNION5)
prodl = {i: PRODUCTS[i].lower() for i in UNION5}
SV_global = {i for i in covered if any(fuzz.token_set_ratio(c, prodl[i]) >= THR for c in _gcands)}

# per-(carrier, doc) span sentences + doc-level localizability
span_map = {}                                       # (carrier_idx, doc) -> [span sentences]
locatable_docs = {i: set() for i in UNION5}         # carrier_idx -> {docs with >=1 span}
for doc in DOCS:
    for s in sentences(DOCTEXT[doc]):
        cs = set()
        for rx in RXS:
            for m in rx.finditer(s):
                cs.add(m.group(0).strip().lower())
        if not cs:
            continue
        for i in UNION5:
            if any(fuzz.token_set_ratio(c, prodl[i]) >= THR for c in cs):
                span_map.setdefault((i, doc), []).append(s)
                locatable_docs[i].add(doc)

recoverable = {i for i in UNION5 if locatable_docs[i]}   # scanner-visible AND span-locatable
print(f"H248 global scanner-visible carriers : {len(SV_global)}/{U}   (expected 55)")
print(f"span-locatable (recoverable) carriers: {len(recoverable)}/{U}")
log(f"scanner SV={len(SV_global)} recoverable={len(recoverable)}")

H248 global scanner-visible carriers : 55/63   (expected 55)
span-locatable (recoverable) carriers: 55/63


## R24-H252 - scanner-audited surgical re-extraction (free clause)

**Registered free clause**: >= 60% of the gold carriers missed by a single arm-A run are scanner-visible AND locatable to sentence spans totaling <= 0.4x the document's chunk tokens. The audit is free (deterministic scanner over chunk text); a micro-pass would re-read only those spans, paying tokens proportional to the residue.

**Method**: for each arm-A run, the miss set is `union-5 minus that run's pooled matches`. The recoverable share is the fraction of missed carriers that are scanner-visible and span-locatable. Token cost is measured per (run, doc): the union of span-sentences for the residue attributed to that document, as a whitespace-token fraction of the document's chunk text.

In [4]:
def ws(t): return len(t.split())
doc_tok = {doc: ws(DOCTEXT[doc]) for doc in DOCS}

# clause A: recoverable share of the per-run miss set
shares = []
for r in RUNS:
    missed = UNION5 - perA[r]
    shares.append(len(missed & recoverable) / len(missed))
h252_share_mean = mean(shares)
h252_share_pooled = (sum(len((UNION5 - perA[r]) & recoverable) for r in RUNS)
                     / sum(len(UNION5 - perA[r]) for r in RUNS))

# clause B: token-cost fraction of the residue spans, per (run, doc)
per_doc_cost = {doc: [] for doc in DOCS}
for r in RUNS:
    missed = UNION5 - perA[r]
    for doc in DOCS:
        res = [i for i in missed if doc in locatable_docs[i]]
        sset = set()
        for i in res:
            sset.update(span_map.get((i, doc), []))
        per_doc_cost[doc].append(ws(" ".join(sset)) / doc_tok[doc] if doc_tok[doc] else 0.0)
doc_cost_mean = {doc: mean(per_doc_cost[doc]) for doc in DOCS}
h252_cost_pooled = mean([c for doc in DOCS for c in per_doc_cost[doc]])
h252_cost_max_doc = max(doc_cost_mean.values())

pass_share = h252_share_mean >= BAR_H252_SHARE
pass_cost = h252_cost_max_doc <= BAR_H252_COST
h252_pass = pass_share and pass_cost

print("H252 residue recoverable share (missed carriers that are scanner-visible + span-locatable):")
for r in RUNS:
    m = UNION5 - perA[r]
    print(f"   run{r}: missed={len(m):2d}  recoverable={len(m & recoverable):2d}  share={len(m & recoverable)/len(m):.2f}")
print(f"   -> mean={h252_share_mean:.3f}  pooled={h252_share_pooled:.3f}  (bar >= {BAR_H252_SHARE}) -> {'PASS' if pass_share else 'FAIL'}")
print("\nH252 residue token-cost fraction per doc (mean over runs), bar <= 0.4:")
for doc in DOCS:
    print(f"   {doc[:44]:44s} {doc_cost_mean[doc]:.3f}")
print(f"   -> pooled mean cost={h252_cost_pooled:.3f}  max-doc={h252_cost_max_doc:.3f}  (bar <= {BAR_H252_COST}) -> {'PASS' if pass_cost else 'FAIL'}")

h252_verdict = "CONFIRMED" if h252_pass else "REFUTED"
print(f"\nH252 free-clause verdict: {h252_verdict}")
RESULTS["H252"] = {
    "clause": "free (scanner-audited residue)",
    "recoverable_share_mean": round(h252_share_mean, 4),
    "recoverable_share_pooled": round(h252_share_pooled, 4),
    "per_run_share": {int(r): round(len((UNION5 - perA[r]) & recoverable)/len(UNION5 - perA[r]), 4) for r in RUNS},
    "bar_share": BAR_H252_SHARE, "pass_share": bool(pass_share),
    "token_cost_pooled_mean": round(h252_cost_pooled, 4),
    "token_cost_max_doc_mean": round(h252_cost_max_doc, 4),
    "per_doc_cost_mean": {doc: round(doc_cost_mean[doc], 4) for doc in DOCS},
    "bar_cost": BAR_H252_COST, "pass_cost": bool(pass_cost),
    "pass": bool(h252_pass), "verdict_recommendation": h252_verdict,
    "scanner_visible": len(SV_global), "recoverable_carriers": len(recoverable), "union5": U,
    "note": "residue is scanner-visible+span-locatable; token cost is the union of residue span-sentences per (run,doc) over doc chunk tokens; both clauses clear their bars",
}
log(f"H252 share_mean={h252_share_mean:.3f} cost_pooled={h252_cost_pooled:.3f} verdict={h252_verdict}")

H252 residue recoverable share (missed carriers that are scanner-visible + span-locatable):
   run1: missed= 8  recoverable= 5  share=0.62
   run2: missed=13  recoverable= 9  share=0.69
   run3: missed=15  recoverable= 9  share=0.60
   run4: missed=27  recoverable=22  share=0.81
   run5: missed=10  recoverable= 7  share=0.70
   -> mean=0.686  pooled=0.712  (bar >= 0.6) -> PASS

H252 residue token-cost fraction per doc (mean over runs), bar <= 0.4:
   0-20190113114505.pdf                         0.223
   1017900r4_ResMed_Product_Catalogue_ANZ_Eng_L 0.046
   3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1 0.004
   ARTP_Standards_of_Care_-_CPAP_Devices_(Techn 0.004
   Airsense-Brochure.pdf                        0.090
   BC-Dreamstation-Standard-CPAP.pdf            0.050
   BMC_RESmart_AutoCPAP_User_Manual.pdf         0.009
   Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf  0.064
   CPAP-Machines-Brochure.pdf                   0.015
   CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf  0.000
   -> po

## R24-H253 - the severed-association linkage (two gaps, one cause?)

**Registered bar**: >= 25% of H226's never-extracted ownership misses name features whose evidence rows sit in H144's severed-row set (same document, feature name `token_set_ratio >= 70` in a severed row's text).

**Method**: the H144 report persists only up to 5 severed-row example texts per document, so the severed rows are recomputed in full from the H144 boundary-audit method (2000/200 cl100k chunker at character offsets; a markdown table row is severed when it lands in a chunk that never contains its header). Recompute is restricted to the documents that carry never-extracted misses. A miss links if its feature name (underscores -> spaces) matches any severed row of the same document at `token_set_ratio >= 70`. A threshold sensitivity sweep tests robustness.

**Corpus caveat**: H144 ran on the 27/28-doc benchmark corpus; H226 ran on the reference graph's corpus. The document overlap for the never-extracted misses is reported explicitly.

In [5]:
enc = tiktoken.get_encoding("cl100k_base")

def chunks_with_spans(doc):
    """H144 chunk_document logic + char-span capture."""
    text = doc.text
    if not text.strip():
        return []
    tokens = enc.encode(text); total = len(tokens)
    if total <= CHUNK_SIZE:
        return [{"index": 0, "cstart": 0, "cend": len(text)}]
    out = []; start = index = 0
    while start < total:
        end = min(start + CHUNK_SIZE, total)
        piece = enc.decode(tokens[start:end])
        if end < total:
            piece = _snap_to_boundary(piece, enc, CHUNK_SIZE)
        if not piece.strip():
            start = end; continue
        tc = len(enc.encode(piece)); cstart = len(enc.decode(tokens[:start]))
        out.append({"index": index, "cstart": cstart, "cend": cstart + len(piece)}); index += 1
        if end >= total:
            break
        start += max(tc - CHUNK_OVERLAP, 1)
    return out

SEP_RE = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$")
ROW_RE = re.compile(r"^\s*\|.*\|\s*$")
def line_spans(text):
    out = []; pos = 0
    for ln in text.splitlines(keepends=True):
        out.append((ln.rstrip("\n"), pos, pos + len(ln.rstrip("\n")))); pos += len(ln)
    return out
def tables(text):
    ls = line_spans(text); out = []; i = 0
    while i < len(ls) - 1:
        htxt, ha, hb = ls[i]; stxt, _, _ = ls[i + 1]
        if ROW_RE.match(htxt) and SEP_RE.match(stxt):
            rows = []; j = i + 2
            while j < len(ls) and ROW_RE.match(ls[j][0]):
                rows.append((ls[j][1], ls[j][2])); j += 1
            out.append({"header": (ha, hb), "rows": rows}); i = j
        else:
            i += 1
    return out
def chunk_of(span, ch):
    a, b = span
    return [c["index"] for c in ch if c["cstart"] <= a and b <= c["cend"]]

# H226 never-extracted misses: (home_doc, product, feature)
h226 = json.load(open(H226_REPORT))
NEVER = [(pp["home_doc"], pp["product"], row["feature"])
         for pp in h226["per_product"] for row in pp["rows"]
         if row.get("mechanism") == "never_extracted"]
never_docs = sorted({d for d, _, _ in NEVER})

# recompute full severed-row text for the never-extracted docs
severed_by_doc = {}
for name in never_docs:
    p = DOC_DIR / name
    if not p.exists():
        severed_by_doc[name] = []; continue
    d = read_document(p); ch = chunks_with_spans(d)
    if len(ch) < 2:
        severed_by_doc[name] = []; continue
    rows = []
    for tbl in tables(d.text):
        hc = set(chunk_of(tbl["header"], ch))
        for r in tbl["rows"]:
            rc = set(chunk_of(r, ch))
            if rc and not (rc & hc):
                rows.append(d.text[r[0]:r[1]])
    severed_by_doc[name] = rows

overlap_docs = [d for d in never_docs if severed_by_doc.get(d)]
print(f"never-extracted misses: {len(NEVER)}  across docs: {never_docs}")
print(f"severed-doc overlap (never-extracted doc WITH severed rows): {overlap_docs}")
print("severed-row recompute per never-extracted doc:")
for d in never_docs:
    print(f"   {d[:48]:48s} severed={len(severed_by_doc.get(d, []))}")

never-extracted misses: 12  across docs: ['3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf', 'BMC_RESmart_AutoCPAP_User_Manual.pdf', 'DreamStation_CPAP_User_Manual.pdf', 'PrismaSmart-and-Soft-Max-Brochure.pdf', 'ResMed-Airsense-11-Manual.pdf']
severed-doc overlap (never-extracted doc WITH severed rows): ['3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf', 'BMC_RESmart_AutoCPAP_User_Manual.pdf', 'DreamStation_CPAP_User_Manual.pdf', 'ResMed-Airsense-11-Manual.pdf']
severed-row recompute per never-extracted doc:
   3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_E severed=3
   BMC_RESmart_AutoCPAP_User_Manual.pdf             severed=3
   DreamStation_CPAP_User_Manual.pdf                severed=13
   PrismaSmart-and-Soft-Max-Brochure.pdf            severed=0
   ResMed-Airsense-11-Manual.pdf                    severed=8


In [6]:
def feat_readable(f): return f.replace("_", " ")

def best_row(feat, doc):
    return max([(fuzz.token_set_ratio(feat_readable(feat), row.lower()), row)
                for row in severed_by_doc.get(doc, [])] or [(0, None)], key=lambda x: x[0])

linked = []
for doc, prod, feat in NEVER:
    score, row = best_row(feat, doc)
    if score >= H253_MATCH_THR:
        linked.append((doc, prod, feat, score, row))
h253_link_share = len(linked) / len(NEVER)
h253_pass = h253_link_share >= BAR_H253_LINK

print(f"H253 linked {len(linked)}/{len(NEVER)} = {h253_link_share:.1%}  (bar >= {BAR_H253_LINK:.0%}) -> {'PASS' if h253_pass else 'FAIL'}")
print("\nlinked misses (feature -> best severed row, with score and shared tokens):")
for doc, prod, feat, score, row in linked:
    shared = set(feat_readable(feat).split()) & set(row.lower().replace("|", " ").replace("<br>", " ").split())
    print(f"   [{doc[:26]}] {prod} :: {feat}  score={score:.0f}  shared={shared or '{}'}")
    print(f"       row: {row[:110]!r}")

# threshold sensitivity - robustness of the linkage
print("\nthreshold sensitivity (linked / 12):")
sens = {}
for th in (60, 65, 70, 75, 80, 85, 90):
    n = sum(1 for doc, _, feat in NEVER if best_row(feat, doc)[0] >= th)
    sens[th] = n
    print(f"   thr={th}: {n}/{len(NEVER)} = {n/len(NEVER):.1%}  {'PASS' if n/len(NEVER) >= BAR_H253_LINK else 'FAIL'}")

# per-document breakdown
byd = Counter(d for d, _, _ in NEVER); byl = Counter(d for d, _, _, _, _ in linked)
print("\nper-document never-extracted / linked:")
for d in byd:
    print(f"   {d[:46]:46s} {byd[d]} / {byl.get(d, 0)}")

# Honest read: the linkage grazes the bar at exactly the registered threshold and every link
# scores 71-73 on a single shared token; it collapses to zero one step above threshold, and one
# link (auto_adjust <-> "adjust the time") is a clear semantic false positive.
fragile = (h253_link_share <= 0.30) and (sens.get(75, 0) / len(NEVER) < BAR_H253_LINK)
h253_verdict = "REFUTED" if fragile else ("CONFIRMED" if h253_pass else "REFUTED")
print(f"\nH253 nominal-pass={h253_pass}  fragile={fragile}  ->  verdict_recommendation: {h253_verdict}")
RESULTS["H253"] = {
    "never_extracted_misses": len(NEVER),
    "never_extracted_docs": never_docs,
    "severed_overlap_docs": overlap_docs,
    "severed_rows_recomputed": {d: len(severed_by_doc.get(d, [])) for d in never_docs},
    "linked": [{"doc": d, "product": p, "feature": f, "score": round(s, 1)} for d, p, f, s, _ in linked],
    "link_share": round(h253_link_share, 4), "bar": BAR_H253_LINK,
    "match_threshold": H253_MATCH_THR,
    "threshold_sensitivity": {str(k): v for k, v in sens.items()},
    "per_doc_never_linked": {d: [byd[d], byl.get(d, 0)] for d in byd},
    "nominal_pass": bool(h253_pass), "fragile": bool(fragile),
    "pass": bool(h253_pass and not fragile),
    "verdict_recommendation": h253_verdict,
    "note": ("linkage lands at exactly 3/12=25.0% at the registered token_set_ratio>=70 but collapses "
             "to 0/12 at threshold 75; all three links score 71-73 on a single shared token "
             "(heated/cellular/adjust) and one (auto_adjust vs 'adjust the time') is a semantic false "
             "positive - the two loss classes separate cleanly, so H145 stays deprioritized"),
}
log(f"H253 linked={len(linked)}/{len(NEVER)}={h253_link_share:.3f} fragile={fragile} verdict={h253_verdict}")

H253 linked 3/12 = 25.0%  (bar >= 25%) -> PASS

linked misses (feature -> best severed row, with score and shared tokens):
   [DreamStation_CPAP_User_Man] DreamStation CPAP :: heated_tube  score=71  shared={'heated'}
       row: '|The airfow pressure<br>feels too high or too<br>low.|The Tubing type<br>setting may be<br>incorrect.|Make sur'
   [DreamStation_CPAP_User_Man] DreamStation CPAP :: cellular_modem  score=73  shared={'cellular'}
       row: '|The device’s display<br>is erratic.|The device has<br>been dropped or<br>mishandled, or<br>the device is in<b'
   [DreamStation_CPAP_User_Man] DreamStation CPAP :: auto_adjust  score=71  shared={'adjust'}
       row: '||Time|Allows you to adjust the time. The default setting is Greenwich Mean Time, but<br>you may adjust the ti'

threshold sensitivity (linked / 12):
   thr=60: 3/12 = 25.0%  PASS
   thr=65: 3/12 = 25.0%  PASS
   thr=70: 3/12 = 25.0%  PASS
   thr=75: 0/12 = 0.0%  FAIL
   thr=80: 0/12 = 0.0%  FAIL
   thr=85: 0/12 = 0.0%  FAIL


## R24-H254 - graph-as-lexicon healing (free ceiling clause)

**Registered ceiling clause**: >= 50% (mean over arm-A runs) of the gold carriers missed by a single run are already present as reference-graph entities extracted from OTHER documents AND their names appear in the missed document's chunk text (`token_set_ratio >= 85`). Also report the first-appearance share - missed carriers the graph knows from no other document, the slice healing cannot reach.

**Method**: pull entity names + `source_documents` from the read-only reference graph and the `KGFDocument` name->id map. "Name appears in the document's chunk text" reuses the H248 scanner span-localization (the same primitive as H252; a raw `token_set_ratio` of a short name against a whole document is diluted and undercounts). A missed carrier is **healable** when it is span-locatable in some document `d` and a matching graph entity carries a `source_documents` id other than `d` - i.e. the graph already learned it elsewhere. First-appearance = the graph knows the carrier from at most one document.

In [7]:
driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
with driver.session() as s:                        # STRICTLY READ-ONLY (MATCH only)
    ents = s.run("MATCH (e:Entity) WHERE e.source_documents IS NOT NULL "
                 "RETURN e.name AS n, e.source_documents AS sd").data()
    docmap = s.run("MATCH (d:KGFDocument) RETURN d.name AS n, d.id AS id").data()
driver.close()
NAME2ID = {r["n"]: r["id"] for r in docmap}
DOCID = {doc: NAME2ID.get(doc) for doc in DOCS}
print(f"graph entities with source_documents: {len(ents)}   docid mapped: {sum(v is not None for v in DOCID.values())}/{len(DOCS)}")

# graph_docs[carrier] = union of source_documents over entities whose name matches the carrier
graph_docs = {i: set() for i in UNION5}
for e in ents:
    en = (e["n"] or "").lower()
    for i in UNION5:
        if fuzz.token_set_ratio(en, prodl[i]) >= THR:
            graph_docs[i].update(e["sd"] or [])

def healable(i):
    for d in locatable_docs[i]:                    # doc where the name appears in chunk text
        if graph_docs[i] - {DOCID[d]}:             # graph knows it from a DIFFERENT doc
            return True
    return False
def first_appearance(i):
    return len(graph_docs[i]) <= 1                 # graph has no other document to heal from

ceil_shares, fa_shares = [], []
for r in RUNS:
    missed = UNION5 - perA[r]
    ceil_shares.append(len([i for i in missed if healable(i)]) / len(missed))
    fa_shares.append(len([i for i in missed if first_appearance(i)]) / len(missed))
h254_ceiling_mean = mean(ceil_shares)
h254_ceiling_pooled = (sum(len([i for i in (UNION5 - perA[r]) if healable(i)]) for r in RUNS)
                       / sum(len(UNION5 - perA[r]) for r in RUNS))
h254_fa_mean = mean(fa_shares)
h254_pass = h254_ceiling_mean >= BAR_H254_CEILING

print(f"\nH254 healing ceiling (healable share of per-run miss set):")
for r in RUNS:
    m = UNION5 - perA[r]
    print(f"   run{r}: missed={len(m):2d}  healable={len([i for i in m if healable(i)]):2d}  "
          f"first_app={len([i for i in m if first_appearance(i)]):2d}  share={len([i for i in m if healable(i)])/len(m):.2f}")
print(f"   -> ceiling mean={h254_ceiling_mean:.3f}  pooled={h254_ceiling_pooled:.3f}  (bar >= {BAR_H254_CEILING}) -> {'PASS' if h254_pass else 'FAIL'}")
print(f"   -> first-appearance share (healing cannot reach) mean={h254_fa_mean:.3f}")

allm = set().union(*[UNION5 - perA[r] for r in RUNS])
fate = {"healable": sum(healable(i) for i in allm),
        "in_graph_multi_doc": sum(len(graph_docs[i]) > 1 for i in allm),
        "in_graph_single_doc_or_absent": sum(len(graph_docs[i]) <= 1 for i in allm),
        "matched_a_graph_entity": sum(bool(graph_docs[i]) for i in allm)}
print(f"\never-missed unique carriers n={len(allm)}: {fate}")

h254_verdict = "CONFIRMED" if h254_pass else "REFUTED"
print(f"\nH254 ceiling-clause verdict: {h254_verdict}")
RESULTS["H254"] = {
    "clause": "free ceiling (graph-as-lexicon)",
    "ceiling_mean": round(h254_ceiling_mean, 4),
    "ceiling_pooled": round(h254_ceiling_pooled, 4),
    "per_run_ceiling": {int(r): round(len([i for i in (UNION5 - perA[r]) if healable(i)])/len(UNION5 - perA[r]), 4) for r in RUNS},
    "bar": BAR_H254_CEILING, "pass": bool(h254_pass),
    "first_appearance_share_mean": round(h254_fa_mean, 4),
    "ever_missed_unique": len(allm), "fate": {k: int(v) for k, v in fate.items()},
    "graph_entities": len(ents),
    "verdict_recommendation": h254_verdict,
    "note": ("'name appears in chunk text' uses the H248 scanner span-localization (a raw token_set_ratio of a "
             "short name vs a whole document is diluted); healable = span-locatable AND graph knows the carrier "
             "from another document; ceiling clears comfortably, ~31% first-appearance slice needs the paid cures"),
}
log(f"H254 ceiling_mean={h254_ceiling_mean:.3f} first_app={h254_fa_mean:.3f} verdict={h254_verdict}")

graph entities with source_documents: 2798   docid mapped: 10/10

H254 healing ceiling (healable share of per-run miss set):
   run1: missed= 8  healable= 3  first_app= 4  share=0.38
   run2: missed=13  healable= 9  first_app= 5  share=0.69
   run3: missed=15  healable= 9  first_app= 4  share=0.60
   run4: missed=27  healable=21  first_app= 6  share=0.78
   run5: missed=10  healable= 7  first_app= 2  share=0.70
   -> ceiling mean=0.629  pooled=0.671  (bar >= 0.5) -> PASS
   -> first-appearance share (healing cannot reach) mean=0.315

ever-missed unique carriers n=31: {'healable': 23, 'in_graph_multi_doc': 24, 'in_graph_single_doc_or_absent': 7, 'matched_a_graph_entity': 30}

H254 ceiling-clause verdict: CONFIRMED


## Conclusions

Grounded in the cell outputs above. Bars are the ones pre-registered in `docs/experiments/kgf-redesign-experiments.md` section R24.

- **H252 (scanner-audited residue) - CONFIRMED**: of the carriers a single arm-A run misses, a mean **68.6%** (pooled 71.2%) are scanner-visible and span-locatable - above the 60% bar - and the residue spans cost a pooled **5.0%** of chunk tokens (worst document 22%, well under the 0.4x bar). The free audit clause holds: a surgical micro-pass has a well-defined, cheap residue to re-read. This gates the LLM clause (queued behind H229)

- **H253 (severed-association linkage) - REFUTED (fragile grazing pass)**: the linkage lands at exactly **3/12 = 25.0%** at the registered `token_set_ratio >= 70`, nominally clearing the 25% bar, but the result is not robust - every link scores 71-73 on a single shared token (heated / cellular / adjust), one link (`auto_adjust` vs a row about adjusting the clock time) is a clear semantic false positive, and raising the threshold five points to 75 collapses linkage to **0/12**. The two loss classes separate cleanly (severance loses table VALUES; the ownership gap loses prose-stated features), so H145 stays deprioritized rather than promoted

- **H254 (graph-as-lexicon ceiling) - CONFIRMED**: a mean **62.9%** (pooled 67.1%) of per-run missed carriers already exist in the reference graph from another document and re-appear in the missed document's chunk text - above the 50% ceiling bar. The remaining **~31.5%** are first-appearance carriers the graph cannot heal, which the paid K-pass cures must still cover. The zero-LLM lexicon scan is a real load-stage recall lever whose reach grows with the graph

**Routing**: H252 and H254 free clauses PASS and earn their LLM/operator clauses in the post-H229 queue; H253 does not survive its robustness check and keeps semantic (SaT/table-atomic) chunking deprioritized against extraction-recall levers.

## Machine-readable report

In [8]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "tier": "R24 free gates",
    "executor": "Claude (opus executor)",
    "stamp": stamp,
    "driver": NEO4J_URI, "mode": "READ-ONLY",
    "bars": {"H252_share": BAR_H252_SHARE, "H252_cost": BAR_H252_COST,
             "H253_link": BAR_H253_LINK, "H254_ceiling": BAR_H254_CEILING},
    "sanity_anchors": {"union5": U, "single_cov": round(single_cov, 4),
                       "scanner_visible": len(SV_global)},
    "results": RESULTS,
}
out = REPORTS_DIR / f"remedies-free-gates-r24-{stamp}.json"
out.write_text(json.dumps(report, indent=2))
print("wrote", out)
for h, res in RESULTS.items():
    print(f"   {h}: {res['verdict_recommendation']}")
log(f"report written {out}")

wrote reports/remedies-free-gates-r24-20260708T070017Z.json
   H252: CONFIRMED
   H253: REFUTED
   H254: CONFIRMED
